### Support Vector Regressor:

##### Data:
The concrete slump test measures the consistency of fresh concrete before it sets. It is performed to check the workability of freshly made concrete, and therefore the ease with which concrete flows. It can also be used as an indicator of an improperly mixed batch.

Our data set consists of various cement properties and the resulting slump test metrics in cm. Later on the set concrete is tested for its compressive strength 28 days later.

Input variables (7)(component kg in one M^3 concrete):
* Cement
* Slag
* Fly ash
* Water
* SP
* Coarse Aggr.
* Fine Aggr.

Output variables (3):
* SLUMP (cm)
* FLOW (cm)
* **28-day Compressive Strength (Mpa)**

#### 1) Importing Necessary Libraries:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

#### 2) Loading Data:

In [2]:
df = pd.read_csv('cement_slump.csv')

In [3]:
df.head()

,Cement,Slag,Fly ash,Water,SP,Coarse Aggr.,Fine Aggr.,SLUMP(cm),FLOW(cm),Compressive Strength (28-day)(Mpa)
0,273.0,82.0,105.0,210.0,9.0,904.0,680.0,23.0,62.0,34.99
1,163.0,149.0,191.0,180.0,12.0,843.0,746.0,0.0,20.0,41.14
2,162.0,148.0,191.0,179.0,16.0,840.0,743.0,1.0,20.0,41.81
3,162.0,148.0,190.0,179.0,19.0,838.0,741.0,3.0,21.5,42.08
4,154.0,112.0,144.0,220.0,10.0,923.0,658.0,20.0,64.0,26.82


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Cement                              103 non-null    float64
 1   Slag                                103 non-null    float64
 2   Fly ash                             103 non-null    float64
 3   Water                               103 non-null    float64
 4   SP                                  103 non-null    float64
 5   Coarse Aggr.                        103 non-null    float64
 6   Fine Aggr.                          103 non-null    float64
 7   SLUMP(cm)                           103 non-null    float64
 8   FLOW(cm)                            103 non-null    float64
 9   Compressive Strength (28-day)(Mpa)  103 non-null    float64
dtypes: float64(10)
memory usage: 8.2 KB


In [5]:
df.shape

(103, 10)

In [6]:
df.describe()

,Cement,Slag,Fly ash,Water,SP,Coarse Aggr.,Fine Aggr.,SLUMP(cm),FLOW(cm),Compressive Strength (28-day)(Mpa)
count,103.000000,103.000000,103.000000,103.000000,103.000000,103.000000,103.000000,103.000000,103.00000,103.000000
mean,229.894175,77.973786,149.014563,197.167961,8.539806,883.978641,739.604854,18.048544,49.61068,36.038738
std,78.877230,60.461363,85.418080,20.208158,2.807530,88.391393,63.342117,8.750844,17.56861,7.837120
min,137.000000,0.000000,0.000000,160.000000,4.400000,708.000000,640.600000,0.000000,20.00000,17.190000
25%,152.000000,0.050000,115.500000,180.000000,6.000000,819.500000,684.500000,14.500000,38.50000,30.900000
50%,248.000000,100.000000,164.000000,196.000000,8.000000,879.000000,742.700000,21.500000,54.00000,35.520000
75%,303.900000,125.000000,235.950000,209.500000,10.000000,952.800000,788.000000,24.000000,63.75000,41.205000
max,374.000000,193.000000,260.000000,240.000000,19.000000,1049.900000,902.000000,29.000000,78.00000,58.530000


- All features are Numeric and Continuous in nature.
- Target Variable is Numeric and Continuous.
- There are no missing values in data.

#### 3) Preprocessing Pipeline:

In [11]:
preprocessor = ColumnTransformer([
    ('Preprocessing', StandardScaler(), list(df.drop('Compressive Strength (28-day)(Mpa)', axis= 1).columns))
])

#### 4) Train Test Split:
- For simplicity of demo, we won't apply any transformations to any features except scaling.

In [12]:
X = df.drop('Compressive Strength (28-day)(Mpa)', axis= 1) #Features
y = df['Compressive Strength (28-day)(Mpa)'] #Target

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

In [14]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(82, 9)
(21, 9)
(82,)
(21,)


#### 5) Grid Search on Support Vector Regressor:

In [16]:
full_pipeline = Pipeline([
    ('Prepocessing', preprocessor),
    ('Model', SVR())
])

In [18]:
param_dict = {'Model__kernel' : ['linear','poly','rbf','sigmoid'],
             'Model__C' : [0.01, 0.1, 1, 10, 100],
             'Model__gamma' : ['scale', 'auto'],
             'Model__degree' : [1,2,3],
             'Model__epsilon' : [0.01, 0.1, 0, 0.5, 1, 2]}

In [20]:
grid_model = GridSearchCV(estimator= full_pipeline,
                         param_grid= param_dict,
                         cv= 5)

In [21]:
# Fitting Grid Model:
grid_model.fit(X_train, y_train)

,estimator,"Pipeline(step...del', SVR())])"
,param_grid,"{'Model__C': [0.01, 0.1, ...], 'Model__degree': [1, 2, ...], 'Model__epsilon': [0.01, 0.1, ...], 'Model__gamma': ['scale', 'auto'], ...}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Preprocessing', ...)]"


In [22]:
# Best Estimator and Parameters according to Grid:

print(grid_model.best_params_)
print(grid_model.best_score_)

{'Model__C': 100, 'Model__degree': 1, 'Model__epsilon': 0, 'Model__gamma': 'scale', 'Model__kernel': 'rbf'}
0.9152069006125764


In [23]:
# Predictions using Best Estimator from Grid:
y_pred = grid_model.predict(X_test)

In [24]:
# Model Evaluation:

print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred))}')
print(f'R2-Score: {r2_score(y_test, y_pred)}')

MAE: 0.9792301074685588
RMSE: 1.470320914593194
R2-Score: 0.957273472792728
